# 04b — SEEG decomposição em sub-canais (B4.M.1 + B4.M.2)

**Pré-registro v2.3.8 §3.10.2, §7.7**

Este notebook:
1. **B4.M.1** — roda o pipeline SEEG refatorado, que agora emite os 6 sub-canais de decomposição do `solos_manejados` **em paralelo** aos 5 macro-canais (Opção 2 — macro-canais intactos).
2. **Auditoria de strings** — antes de confiar na classificação, lista as `(Sub-categoria, Produto)` REAIS dentro de `solos_manejados` no SEEG, para confirmar que as strings canônicas do `seeg.py` batem. **Esta é a verificação que torna o refator seguro.**
3. **B4.M.2** — valida a identidade algébrica `Σ sub-canais ≡ solos_manejados` ao 4º decimal.

**Pré-condição:** `seeg.py` refatorado já subido em `MyDrive/Renovabio - EcoEco/pipeline/seeg.py`.

**O que NÃO muda:** os 5 macro-canais (`luc`, `carbono_solo`, `queima`, `solos_manejados`, `residuos_florestais`), a auditoria F3, o balanceamento macro, os outcomes já validados em v2.3.7. Tudo preservado.

In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path
BASE_DIR = Path('/content/drive/MyDrive/Renovabio - EcoEco')
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
print('setup ok')

In [ ]:
# Reload módulos (garante que pega o seeg.py refatorado recém-subido)
import importlib
from pipeline import config, io, seeg
importlib.reload(config)
importlib.reload(io)
importlib.reload(seeg)

from pipeline.config import PARAMS, interim, out_pre
from pipeline.seeg import (
    run_seeg_pipeline, classify_channel, classify_subchannel,
    validate_subchannel_algebra,
    SUBCANAIS_SOLOS, SUBCANAL_TRANSFORM,
    SUBCAT_RESIDUOS_AGRICOLAS, SUBCAT_RESIDUOS_ORGANICOS,
    SUBCAT_FERTILIZANTES_N, SUBCAT_CORRETIVO,
    PROD_CANA, PRODS_ORGANICOS_CANA, PROD_CALAGEM,
)
print('módulos carregados — seeg.py refatorado v2.3.8')
print('sub-canais declarados:', SUBCANAIS_SOLOS)
print('transformações:', SUBCANAL_TRANSFORM)

In [ ]:
# Carrega crosswalk e ANP (para auditoria F3 — idêntico ao notebook 03)
cw = pd.read_csv(interim('crosswalk_centrosul.csv'), dtype={'geocode': str})
muni_treat = pd.read_csv(interim('anp_muni_treat.csv'), dtype={'geocode': str})
cana_baseline_munis = muni_treat['geocode'].unique().tolist()

print(f'Crosswalk: {cw.shape}')
print(f'Municípios canavieiros via ANP: {len(cana_baseline_munis)}')

## Auditoria de strings — ANTES de confiar na classificação

O `classify_subchannel` depende de strings exatas de `Sub-categoria emissora` e `Produto ou sistema`. Se alguma divergir do SEEG real, a partição vaza para `res_minor` e a decomposição fica errada (mas a identidade algébrica AINDA fecha — por isso a auditoria é necessária *além* da validação).

Esta célula lê uma UF de amostra (SP, a maior) e lista TODAS as `(Sub-categoria, Produto)` distintas dentro de `solos_manejados`.

In [ ]:
# Auditoria de strings reais dentro de solos_manejados (amostra: SP)
from pipeline.seeg import (
    read_seeg_uf, filter_afolu, filter_emission_signal, filter_co2e_gtp,
)

df_sp = read_seeg_uf('SP')
df_sp = filter_afolu(df_sp)
df_sp = filter_emission_signal(df_sp)
df_sp = filter_co2e_gtp(df_sp)

# Aplica classify_channel para isolar linhas de solos_manejados
df_sp = df_sp.copy()
df_sp['_macro'] = df_sp.apply(classify_channel, axis=1)
sm = df_sp[df_sp['_macro'] == 'solos_manejados'].copy()

print(f'Linhas solos_manejados (SP): {len(sm):,}')
print('\n=== (Sub-categoria emissora, Produto ou sistema) distintas ===\n')
combo = (sm.groupby(['Sub-categoria emissora', 'Produto ou sistema'])
           .size().reset_index(name='n_linhas')
           .sort_values('n_linhas', ascending=False))
pd.set_option('display.max_rows', 100)
print(combo.to_string(index=False))

In [ ]:
# Confronto: as strings canônicas do seeg.py existem no SEEG real?
subcats_reais = set(sm['Sub-categoria emissora'].dropna().unique())
prods_reais = set(sm['Produto ou sistema'].dropna().unique())

print('=== Sub-categorias canônicas vs SEEG real ===')
for nome, val in [
    ('RESIDUOS_AGRICOLAS', SUBCAT_RESIDUOS_AGRICOLAS),
    ('RESIDUOS_ORGANICOS', SUBCAT_RESIDUOS_ORGANICOS),
    ('FERTILIZANTES_N',    SUBCAT_FERTILIZANTES_N),
    ('CORRETIVO',          SUBCAT_CORRETIVO),
]:
    achou = val in subcats_reais
    print(f'  {"OK " if achou else "XX "} {nome:22s} = {val!r}  '
          f'{"existe" if achou else "*** NAO ENCONTRADO ***"}')

print('\n=== Produtos canônicos vs SEEG real ===')
for nome, val in [('PROD_CANA', PROD_CANA), ('PROD_CALAGEM', PROD_CALAGEM)]:
    achou = val in prods_reais
    print(f'  {"OK " if achou else "XX "} {nome:22s} = {val!r}  '
          f'{"existe" if achou else "*** NAO ENCONTRADO ***"}')
for p in PRODS_ORGANICOS_CANA:
    achou = p in prods_reais
    print(f'  {"OK " if achou else "XX "} ORGANICO_CANA          = {p!r}  '
          f'{"existe" if achou else "*** NAO ENCONTRADO ***"}')

print('\nSe algum aparecer *** NAO ENCONTRADO ***, NÃO prossiga:')
print('ajuste a string correspondente no topo do seeg.py e re-suba.')

In [ ]:
# Sanity: classify_subchannel cobre 100% das linhas de solos_manejados?
sm['_sub'] = sm.apply(classify_subchannel, axis=1)
n_none = sm['_sub'].isna().sum()
print(f'Linhas solos_manejados sem sub-canal (deve ser 0): {n_none}')
print('\nDistribuição por sub-canal (SP, contagem de linhas):')
print(sm['_sub'].value_counts(dropna=False).to_string())
if n_none == 0:
    print('\nOK: partição exaustiva — toda linha de solos_manejados tem sub-canal.')
else:
    print('\n*** ATENÇÃO: há linhas sem sub-canal — investigar antes de prosseguir.')

## B4.M.1 — Rodar pipeline SEEG refatorado completo

`RERUN = True` força reprocessamento das 6 UFs (~3min). O pipeline já valida a identidade algébrica internamente (B4.M.2) e **aborta se falhar**.

In [ ]:
RERUN = True   # B4.M.1 precisa reprocessar para gerar os sub-canais

result = run_seeg_pipeline(cw, cana_baseline_munis=cana_baseline_munis, save=True)
panel = result['panel']
algebra = result['subchannel_algebra']
print(f'\npanel final: {panel.shape}')
print(f'colunas: {sorted(panel.columns.tolist())}')

## B4.M.2 — Validação algébrica explícita

O pipeline já valida internamente, mas re-executamos aqui de forma visível e geramos um relatório legível, conforme exigido pelo pré-registro §7.7.

In [ ]:
res = validate_subchannel_algebra(panel, tol=1e-4)

print('=' * 56)
print('B4.M.2 — VALIDAÇÃO ALGÉBRICA')
print('  Σ(res_cana,org_cana,fert_n,calagem,res_outros,res_minor)')
print('    ≡ solos_manejados   (tolerância 1e-4 tCO2e)')
print('=' * 56)
print(f"  passou           : {res['ok']}")
print(f"  células testadas : {res['n_cells']:,}")
print(f"  LACUNA excluídas : {res['n_excluded_nan']:,} (NaN por design)")
print(f"  max |Σsub − sm|  : {res['max_abs_diff']:.3e} tCO2e")
if res['ok']:
    print('\n  ✓ IDENTIDADE CONFIRMADA — decomposição é aditivamente exata.')
else:
    print('\n  ✗ FALHOU — top discrepâncias:')
    print(res['worst'].to_string(index=False))
    raise AssertionError('Validação B4.M.2 falhou — não prosseguir.')

In [ ]:
# Verificação adicional: shares empíricos dos sub-canais vs pré-registro §3.10.2
# (esperado aprox: res_cana 5.3%, org_cana 4.1%, fert_n 27.3%,
#  calagem 24.2%, res_outros 31.0%, res_minor ~8.1%)
anos_main = list(range(PARAMS.YEAR_MIN_MAIN, PARAMS.YEAR_MAX_MAIN + 1))
pm = panel[panel['ano'].isin(anos_main)]

tot = pm[list(SUBCANAIS_SOLOS)].sum().sum()
print('Share empírico de cada sub-canal no solos_pipeline (CS 2015-2024):')
print('-' * 52)
ref = {'res_cana': 5.3, 'org_cana': 4.1, 'fert_n': 27.3,
       'calagem': 24.2, 'res_outros': 31.0, 'res_minor': 8.1}
for c in SUBCANAIS_SOLOS:
    share = 100 * pm[c].sum() / tot if tot > 0 else 0
    r = ref.get(c, float('nan'))
    print(f'  {c:12s}: {share:5.1f}%   (pré-reg §3.10.2: ~{r:.1f}%)')
print('-' * 52)
print('Divergências moderadas são esperadas (shares do pré-registro')
print('vieram do diagnóstico D1, amostra e janela podem diferir).')
print('Divergências GRANDES (>2x) sugerem erro de string — investigar.')

In [ ]:
# Sanity final: macro-canais NÃO mudaram (regressão vs v2.3.7)
# solos_manejados deve continuar idêntico ao que era antes do refator.
macro = ['luc', 'carbono_solo', 'queima', 'solos_manejados',
         'residuos_florestais']
print('Macro-canais presentes no painel (devem ser os 5 originais):')
for m in macro:
    pres = m in panel.columns
    print(f'  {"OK " if pres else "XX "} {m}')

print('\nEstatísticas de solos_manejados (referência p/ comparar com 03_seeg):')
sm_stats = panel['solos_manejados'].describe()
print(sm_stats.to_string())
print('\nCompare estes números com a saída do notebook 03_seeg ANTES')
print('do refator. Devem ser IDÊNTICOS (Opção 2 não toca macro-canais).')

## Conclusão B4.M.1 + B4.M.2

Se todas as células acima passaram:

- ✓ **B4.M.1**: pipeline emite 6 sub-canais em paralelo aos 5 macro-canais
- ✓ **Auditoria strings**: strings canônicas batem com o SEEG real
- ✓ **Partição exaustiva**: toda linha de `solos_manejados` tem exatamente um sub-canal
- ✓ **B4.M.2**: `Σ sub ≡ solos_manejados` ao 4º decimal
- ✓ **Regressão**: macro-canais inalterados (Opção 2)

**Artefatos gerados** (em `data/interim/` e `outputs_pre/`):
- `seeg_subcanais_panel.csv` — painel município×ano com 6 sub-canais + transformações + flags
- `seeg_subcanais_validacao_algebrica.csv` — relatório B4.M.2
- `seeg_outcomes_audited.csv` — painel completo (macro+sub), agora com colunas novas

**Próximo passo:** B4.M.3 (construir `share_cana_eq52` pré-2018 da PAM 1612). Não inicie sem confirmar com o pré-registro v2.3.8 §9.6.